# Project 65 — Local Hallucination Audit Notebook

**Detect unsupported claims in LLM-generated text — locally.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (qwen3:8b) |
| Framework | LangChain |
| Interface | Jupyter |

## 1 · What You Will Learn

1. How to detect **hallucinated claims** in LLM outputs
2. Implement a **claim extraction + verification** pipeline
3. Classify claims as **supported, unsupported, or unverifiable**
4. Compute a **hallucination rate** metric

## 2 · Why This Project Matters

LLMs confidently state wrong things. A hallucination audit identifies
unsupported claims before they reach users — essential for RAG and summarization.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports

In [ ]:
import json
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Test Cases with Known Hallucinations

In [ ]:
TEST_CASES = [
    {'id': 'case_1',
     'source': 'Acme Corp reported $4.2B revenue in 2023, up 15% YoY. 12,000 employees across 8 countries.',
     'generated': 'Acme Corp achieved $4.2B revenue in 2023 (+15%). They employ ~15,000 people and are HQd in San Francisco.',
     'known_halls': ['15,000 (source: 12,000)', 'SF HQ (not in source)']},
    {'id': 'case_2',
     'source': 'Python was created by Guido van Rossum in 1991. It emphasizes code readability via indentation.',
     'generated': 'Python, by Guido van Rossum (1991), is the most popular language worldwide, maintained by the PSF.',
     'known_halls': ['most popular (not in source)', 'PSF (not in source)']},
    {'id': 'case_3',
     'source': 'The study found exercise reduces heart disease risk by 30%. Participants exercised 150min/week for 2 years.',
     'generated': 'Exercise (150min/week, 2 years) reduced heart disease risk by 30%. Researchers also noted mental health improvements.',
     'known_halls': ['mental health (not in source)']},
]
print(f'{len(TEST_CASES)} test cases with known hallucinations')

## 6 · Hallucination Detection Pipeline

In [ ]:
def extract_claims(text):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Extract factual claims as a JSON list: ["claim1", "claim2", ...]. ONLY the list.'),
        ('human', '{text}'),
    ])
    raw = (prompt | llm | StrOutputParser()).invoke({'text': text})
    try:
        s, e = raw.find('['), raw.rfind(']') + 1
        if s >= 0: return json.loads(raw[s:e])
    except: pass
    return [text]

def verify_claim(claim, source):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Fact-check claim against source. Return JSON: {"verdict": "supported"/"unsupported"/"unverifiable", "evidence": "..."}'),
        ('human', 'Source: {source}\nClaim: {claim}'),
    ])
    raw = (prompt | llm | StrOutputParser()).invoke({'source': source, 'claim': claim})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        if s >= 0: return json.loads(raw[s:e])
    except: pass
    return {'verdict': 'unverifiable', 'evidence': 'parse error'}

def audit_text(source, generated):
    claims = extract_claims(generated)
    results = [{'claim': c, **verify_claim(c, source)} for c in claims]
    total = len(results)
    unsupported = sum(1 for r in results if r.get('verdict') != 'supported')
    return {'claims': results, 'total': total, 'unsupported': unsupported,
            'hall_rate': round(unsupported/total, 3) if total else 0}

print('Audit pipeline ready.')

## 7 · Run the Audit

In [ ]:
audit_results = []
for tc in TEST_CASES:
    print(f'Auditing: {tc["id"]}...')
    result = audit_text(tc['source'], tc['generated'])
    result['case_id'] = tc['id']
    audit_results.append(result)
    print(f'  Claims: {result["total"]}, Unsupported: {result["unsupported"]}, Rate: {result["hall_rate"]:.0%}')
    for cr in result['claims']:
        icon = {'supported': 'OK', 'unsupported': 'HALL', 'unverifiable': '??'}.get(cr.get('verdict',''), '?')
        print(f'    [{icon}] {cr["claim"][:70]}')

## 8 · Summary

In [ ]:
rows = [{'case': r['case_id'], 'total': r['total'], 'unsupported': r['unsupported'],
         'hall_rate': f'{r["hall_rate"]:.0%}'} for r in audit_results]
print(pd.DataFrame(rows).to_string(index=False))

## 9 · Save Results

In [ ]:
with open('hallucination_audit.json', 'w') as f:
    json.dump(audit_results, f, indent=2)
print('Saved.')

## 10 · Limitations & Improvements

- LLM verifier can also hallucinate
- No NLI model for precise entailment
- Binary verdicts miss partial support
- Add severity scoring and claim importance weighting

## 11 · Key Takeaways

- Hallucination detection = claim extraction + source verification
- Hallucination rate is a measurable quality metric
- Even the verifier can err — use with human review for critical tasks
- Local Ollama makes auditing fast and free